# Modelo exploratório de previsibilidade de assinatura

Este notebook é um **modelo de desenvolvimento e entendimento**, não um resultado final de produção. Ele implementa as camadas definidas para o projeto:

1. auditoria e tratamento do dataset;
2. reconstrução da jornada e cadeia de Markov como baseline;
3. modelo individual de propensão à assinatura, quando a base permitir;
4. camada temporal exploratória, com duração e riscos discretos quando houver dados adequados;
5. exemplo de priorização comercial, sem fixar limiares arbitrários.

O notebook não cria dados sintéticos e não inventa valores. Quando uma junção, interpretação ou estimativa não pode ser confirmada pela estrutura do Excel, ela é marcada como limitação.

## Princípios de uso

- As previsões devem usar somente informações disponíveis até uma data de corte.
- A base de transições e a base de perfil devem ser auditadas antes de qualquer interpretação.
- A cadeia de Markov é um baseline; ela não substitui a modelagem individual.
- O modelo temporal abaixo é exploratório. Sua adoção final depende da qualidade das datas e da definição confirmada dos eventos.
- Resultados por marca ou segmento só devem ser interpretados quando houver volume adequado.
- Associação entre visitas, consultor, especialista e conversão não será tratada como causalidade.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

# Se o notebook for movido, altere apenas esta célula.
DATASET = Path('../dados/Proposta-de-Projeto-Base-Funil.xlsx')
OUTPUT_DIR = Path('resultados-v1')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Dataset:', DATASET)
print('Existe:', DATASET.exists())

Dataset: ..\dados\Proposta-de-Projeto-Base-Funil.xlsx
Existe: True


In [2]:
# Auditoria estrutural: nenhuma semântica é assumida antes da inspeção.
book = pd.ExcelFile(DATASET)
print('Planilhas:', book.sheet_names)

audit_rows = []
raw = {}
for sheet in book.sheet_names:
    frame = pd.read_excel(DATASET, sheet_name=sheet)
    raw[sheet] = frame
    audit_rows.append({
        'planilha': sheet,
        'linhas': len(frame),
        'colunas': len(frame.columns),
        'duplicatas_linha': int(frame.duplicated().sum()),
        'celulas_nulas': int(frame.isna().sum().sum()),
        'colunas': len(frame.columns),
    })
    print(f"\n--- {sheet}: {frame.shape} ---")
    print(list(frame.columns))
    display(frame.head(3))

audit = pd.DataFrame(audit_rows)
display(audit)

Planilhas: ['base-funil-2026', 'sequência-etapas-funil', 'base-escolas', 'base-histórico-etapas', 'base-atuação-especialista', 'base-campanhas-realizadas', 'base-perdas']

--- base-funil-2026: (5900, 9) ---
['fFunil Detalhado Histórico[Id]', 'fFunil Detalhado Histórico[OpportunityId]', 'fFunil Detalhado Histórico[CreatedDate]', 'fFunil Detalhado Histórico[NewValue]', 'fFunil Detalhado Histórico[fOportunidade.AccountId]', 'fFunil Detalhado Histórico[fOportunidade.Tipo_de_sistema__c]', 'fFunil Detalhado Histórico[Data]', 'fFunil Detalhado Histórico[Status Atual]', '[SumfOportunidade_Receita_no_Funil]']


,fFunil Detalhado Histórico[Id],fFunil Detalhado Histórico[OpportunityId],fFunil Detalhado Histórico[CreatedDate],fFunil Detalhado Histórico[NewValue],fFunil Detalhado Histórico[fOportunidade.AccountId],fFunil Detalhado Histórico[fOportunidade.Tipo_de_sistema__c],fFunil Detalhado Histórico[Data],fFunil Detalhado Histórico[Status Atual],[SumfOportunidade_Receita_no_Funil]
0,017N400006nwJ7PIAU,006N400000UsiPYIAZ,2025-10-23 12:56:36,Solicitação de material,0016g00000CxYEIAA3,Poliedro,2025-10-23,Avançou para a Próxima Etapa,1190265.28
1,017N400006ToombIAB,006N400000TXKHAIA5,2025-09-19 13:17:12,Solicitação de material,0014v00002Nso0WAAR,Poliedro,2025-09-19,Avançou para a Próxima Etapa,2148178.10
2,017N400007g8phpIAA,006N400000YQ5JvIAL,2026-01-22 15:02:51,Solicitação de material,001N400000FC3MVIA1,Poliedro,2026-01-22,Perdido na Etapa,667690.99



--- sequência-etapas-funil: (14, 5) ---
['#', 'Fase', 'fFunil Detalhado', 'Sequência Funil', 'Observação']


,#,Fase,fFunil Detalhado,Sequência Funil,Observação
0,1,Análise de Material,Solicitação de material,1.Solicitação de material,NaN
1,2,Análise de Material,Material entregue,2.Material entregue,NaN
2,3,Análise de Material,Validação pedagógica,3.Validação pedagógica,NaN



--- base-escolas: (34642, 20) ---
['IdConta__c', 'Perfil da Escola', 'Porte Censo', 'Cluster', 'Score', 'Prioridade', 'EFAF_Censo__c', 'EFAI_Censo__c', 'EI_Censo__c', 'EM_Censo__c', 'BillingCity', 'BillingStateCode', 'Origem_do_cliente__c', 'Religiao_da_Escola__c', 'Rede__c', 'ID Consultor Comercial', 'Mensalidade_media__c', 'Vencimento de Contrato', 'Sistema de Ensino', 'Soma de Visitas Concluídas']


...


--- base-campanhas-realizadas: (1077, 3) ---
['ID da Conta', 'Campanha Pai', 'Nome da Campanha']


,ID da Conta,Campanha Pai,Nome da Campanha
0,0014v00002cT6CTAA0,Relatórios de Inteligência de Mercado 2026-2027,Relatório Enem
1,0014v00002dpYIoAAM,Relatórios de Inteligência de Mercado 2026-2027,Relatório Enem
2,0014v00002drJ59AAE,Relatórios de Inteligência de Mercado 2026-2027,Relatório Enem



--- base-perdas: (1605, 3) ---
['AccountId', 'Data da Perda', 'Motivo Perda']


,AccountId,Data da Perda,Motivo Perda
0,0014v00002cRai4AAC,5/20/2026,NaN
1,0014v00002cRaloAAC,6/8/2026,Satisfação com preço e serviço do atual parceiro
2,0014v00002cRaloAAC,6/8/2026,Escola não adota sistema de ensino


,planilha,linhas,colunas,duplicatas_linha,celulas_nulas
0,base-funil-2026,5900,9,0,18
1,sequência-etapas-funil,14,5,0,11
2,base-escolas,34642,20,0,87821
3,base-histórico-etapas,6,3,0,0
4,base-atuação-especialista,1623,3,0,0
5,base-campanhas-realizadas,1077,3,0,0
6,base-perdas,1605,3,74,761


## 1. Padronização das planilhas

As funções seguintes apenas renomeiam os campos observados no dataset atual. Elas falham de forma explícita se uma coluna necessária não estiver presente. As demais colunas permanecem preservadas. A base de escolas e as bases auxiliares são agregadas antes das junções para evitar multiplicar linhas da oportunidade.

In [3]:
def require_columns(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f'{label}: colunas ausentes: {missing}')

# Base histórica do funil
funil = raw['base-funil-2026'].copy()
funil_cols = {
    'fFunil Detalhado Histórico[Id]': 'event_id',
    'fFunil Detalhado Histórico[OpportunityId]': 'opportunity_id',
    'fFunil Detalhado Histórico[CreatedDate]': 'created_at',
    'fFunil Detalhado Histórico[NewValue]': 'stage',
    'fFunil Detalhado Histórico[fOportunidade.AccountId]': 'account_id',
    'fFunil Detalhado Histórico[fOportunidade.Tipo_de_sistema__c]': 'brand',
    'fFunil Detalhado Histórico[Data]': 'event_date',
    'fFunil Detalhado Histórico[Status Atual]': 'event_status',
    '[SumfOportunidade_Receita_no_Funil]': 'potential_revenue',
}
require_columns(funil, list(funil_cols), 'base-funil-2026')
funil = funil.rename(columns=funil_cols)
for c in ['created_at', 'event_date']:
    funil[c] = pd.to_datetime(funil[c], errors='coerce')
funil['potential_revenue'] = pd.to_numeric(funil['potential_revenue'], errors='coerce')

# Sequência oficial: usada para documentar estados e ordenar quando possível.
seq = raw['sequência-etapas-funil'].copy()
seq_map_cols = {'#': 'stage_order', 'Fase': 'phase', 'fFunil Detalhado': 'stage', 'Sequência Funil': 'stage_sequence', 'Observação': 'note'}
require_columns(seq, list(seq_map_cols), 'sequência-etapas-funil')
seq = seq.rename(columns=seq_map_cols)
seq['stage_order'] = pd.to_numeric(seq['stage_order'], errors='coerce')

# Perfil das escolas
escolas = raw['base-escolas'].copy()
escola_cols = {
    'IdConta__c':'account_id', 'Perfil da Escola':'school_profile', 'Porte Censo':'school_size',
    'Cluster':'cluster', 'Score':'score', 'Prioridade':'priority', 'EFAF_Censo__c':'students_efaf',
    'EFAI_Censo__c':'students_efai', 'EI_Censo__c':'students_ei', 'EM_Censo__c':'students_em',
    'BillingCity':'city', 'BillingStateCode':'state', 'Origem_do_cliente__c':'lead_source',
    'Religiao_da_Escola__c':'religion', 'Rede__c':'network', 'ID Consultor Comercial':'consultant_id',
    'Mensalidade_media__c':'average_tuition', 'Vencimento de Contrato':'contract_expiry',
    'Sistema de Ensino':'current_system', 'Soma de Visitas Concluídas':'completed_visits'
}
require_columns(escolas, list(escola_cols), 'base-escolas')
escolas = escolas.rename(columns=escola_cols)

# Auxiliares: agregação explícita por chave antes da junção.
esp = raw['base-atuação-especialista'].copy().rename(columns={'AccountId':'account_id','OpportunityId':'opportunity_id','Apresentação de Especialista':'specialist_action'})
esp_summary = esp.groupby('opportunity_id', dropna=False).agg(
    specialist_records=('specialist_action','size'),
    specialist_actions=('specialist_action', lambda s: ' | '.join(sorted(set(s.dropna().astype(str)))))
).reset_index()

camp = raw['base-campanhas-realizadas'].copy().rename(columns={'ID da Conta':'account_id','Campanha Pai':'parent_campaign','Nome da Campanha':'campaign_name'})
camp_summary = camp.groupby('account_id', dropna=False).agg(
    campaign_count=('campaign_name','size'),
    campaign_parents=('parent_campaign', lambda s: ' | '.join(sorted(set(s.dropna().astype(str)))))
).reset_index()

loss = raw['base-perdas'].copy().rename(columns={'AccountId':'account_id','Data da Perda':'loss_date','Motivo Perda':'loss_reason'})
loss['loss_date'] = pd.to_datetime(loss['loss_date'], errors='coerce')
loss_summary = loss.groupby('account_id', dropna=False).agg(
    loss_records=('account_id','size'),
    loss_date_aux=('loss_date','max'),
    loss_reason_aux=('loss_reason', lambda s: ' | '.join(sorted(set(s.dropna().astype(str)))))
).reset_index()

print('Funil padronizado:', funil.shape)
print('Escolas:', escolas.shape, '| especialistas agregados:', esp_summary.shape, '| campanhas:', camp_summary.shape, '| perdas:', loss_summary.shape)

Funil padronizado: (5900, 9)
Escolas: (34642, 20) | especialistas agregados: (1623, 3) | campanhas: (861, 3) | perdas: (1391, 4)


## 2. Reconstrução da jornada

A jornada é reconstruída ordenando os eventos por oportunidade e data. O notebook distingue o estado registrado no histórico do status do evento. Isso permite avaliar se há recuos, saltos ou registros repetidos, sem presumir que todos os eventos sejam transições válidas.

In [4]:
# Enriquecimento do histórico com a ordem oficial das etapas, sem descartar estados não mapeados.
funil = funil.merge(seq[['stage','stage_order','phase']], on='stage', how='left', validate='many_to_one')
funil = funil.sort_values(['opportunity_id','event_date','created_at','event_id'], kind='stable')

journeys = (funil.groupby('opportunity_id', dropna=False)
    .agg(account_id=('account_id','first'), brand=('brand','first'), potential_revenue=('potential_revenue','max'),
         created_at=('created_at','min'), first_event_date=('event_date','min'), last_event_date=('event_date','max'),
         stage_sequence=('stage', list), status_sequence=('event_status', list), event_count=('event_id','size'))
    .reset_index())
journeys['current_stage'] = journeys['stage_sequence'].str[-1]
journeys['unique_stage_count'] = journeys['stage_sequence'].apply(lambda x: len(set(x)))
journeys['transition_count'] = journeys['stage_sequence'].apply(lambda x: max(len(x)-1, 0))
journeys['duration_days'] = (journeys['last_event_date'] - journeys['first_event_date']).dt.days

ABSORBING_WON = 'Contrato assinado'
ABSORBING_LOST = 'Oportunidade perdida'
ABSORBING = [ABSORBING_WON, ABSORBING_LOST]
journeys['outcome'] = np.select([journeys['current_stage'].eq(ABSORBING_WON), journeys['current_stage'].eq(ABSORBING_LOST)], ['won','lost'], default='open')

print('Oportunidades:', len(journeys))
display(journeys['outcome'].value_counts(dropna=False).rename_axis('outcome').to_frame('count'))
display(journeys.head(5))

# Diagnósticos de qualidade da jornada.
journeys['has_missing_dates'] = journeys[['first_event_date','last_event_date']].isna().any(axis=1)
print('Oportunidades com datas ausentes:', int(journeys['has_missing_dates'].sum()))
print('Eventos com etapa sem correspondência na sequência oficial:', int(funil['stage_order'].isna().sum()))
print('Linhas duplicadas por opportunity_id + event_date + stage:', int(funil.duplicated(['opportunity_id','event_date','stage']).sum()))

Oportunidades: 1610


,count
outcome,
lost,897
open,657
won,56


,opportunity_id,account_id,brand,potential_revenue,created_at,first_event_date,last_event_date,stage_sequence,status_sequence,event_count,current_stage,unique_stage_count,transition_count,duration_days,outcome
0,006N400000AZoSRIA1,001N400000C8XpZIAV,Polígono,161441.00,2026-03-11 02:09:23,2026-03-11,2026-04-27,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",3,Validação pedagógica,3,2,47,open
1,006N400000C3EFeIAN,001N400000DxyByIAJ,Poliedro,949049.12,2026-03-17 18:32:40,2026-03-17,2026-06-15,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",4,Oportunidade perdida,4,3,90,lost
2,006N400000KkLAnIAN,0014v00002uO7OMAA0,Poliedro,595118.69,2026-03-04 15:31:14,2026-03-04,2026-03-18,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",12,Contrato assinado,12,11,14,won
3,006N400000Lv4yrIAB,001N400000NlDJqIAN,Poliedro,0.00,2026-08-14 15:24:45,2026-08-14,2026-08-14,[Solicitação de material],[Em Negociação],1,Solicitação de material,1,0,0,open
4,006N400000MWOfOIAX,001N400000AbqbNIAR,Polígono,90706.92,2026-04-28 23:26:56,2026-04-28,2026-06-25,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",4,Oportunidade perdida,4,3,58,lost


Oportunidades com datas ausentes: 0
Eventos com etapa sem correspondência na sequência oficial: 0
Linhas duplicadas por opportunity_id + event_date + stage: 0


## 3. Camada 2 — Cadeia de Markov como baseline

A cadeia é estimada somente com oportunidades finalizadas. Os estados intermediários são transientes, e assinatura e perda são absorventes. A matriz mostra as probabilidades empíricas de transição. Em seguida, a matriz fundamental calcula a probabilidade final de cada absorção a partir de cada estado.

Esta é uma referência simples. Ela não usa o perfil individual da escola nem o tempo em dias na previsão individual.

In [5]:
# Estados observados na sequência oficial, preservando a ordem documentada.
transient_states = [s for s in seq.sort_values('stage_order')['stage'].dropna().tolist() if s not in ABSORBING and s != 'Contrato Ativo']
observed_extra = [s for s in funil['stage'].dropna().unique().tolist() if s not in transient_states + ABSORBING + ['Contrato Ativo']]
all_states = transient_states + observed_extra + ABSORBING

finished = journeys[journeys['outcome'].isin(['won','lost'])].copy()
open_ = journeys[journeys['outcome'].eq('open')].copy()

pairs = []
for stages in finished['stage_sequence']:
    clean = [s for s in stages if s != 'Contrato Ativo' and pd.notna(s)]
    pairs.extend(zip(clean[:-1], clean[1:]))
pairs_df = pd.DataFrame(pairs, columns=['from_state','to_state'])
counts = pd.crosstab(pairs_df['from_state'], pairs_df['to_state']).reindex(index=all_states, columns=all_states, fill_value=0)
transition = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
for state in ABSORBING:
    if state in transition.index:
        transition.loc[state,:] = 0.0
        transition.loc[state,state] = 1.0

print('Finalizadas:', len(finished), '| abertas:', len(open_), '| pares de transição:', len(pairs_df))
display(transition.round(3))
print('Somas das linhas:')
display(transition.sum(axis=1).round(6).to_frame('row_sum'))

# Probabilidade de absorção, se houver estados transientes com transições observadas.
valid_transient = [s for s in transient_states if s in transition.index and transition.loc[s].sum() > 0]
if valid_transient:
    Q = transition.loc[valid_transient, valid_transient].to_numpy()
    R = transition.loc[valid_transient, ABSORBING].to_numpy()
    try:
        N = np.linalg.inv(np.eye(len(valid_transient)) - Q)
        B = N @ R
        absorption = pd.DataFrame(B, index=valid_transient, columns=ABSORBING)
        expected_steps = pd.Series(N.sum(axis=1), index=valid_transient, name='expected_transitions')
        display(absorption.round(4))
        display(expected_steps.round(2).to_frame())
    except np.linalg.LinAlgError:
        absorption = pd.DataFrame()
        expected_steps = pd.Series(dtype=float)
        print('Matriz fundamental não invertível; não foi produzida absorção.')
else:
    absorption = pd.DataFrame()
    expected_steps = pd.Series(dtype=float)
    print('Não há estados transientes válidos para calcular absorção.')

Finalizadas: 953 | abertas: 657 | pares de transição: 2838


to_state,Solicitação de material,Material entregue,Validação pedagógica,Proposta tabela,Proposta personalizada,Negociação e ajuste financeiro,Documentação enviada,Análise de operações comerciais,Análise do Financeiro,Análise do Jurídico,Enviado para a assinatura,Contrato assinado,Oportunidade perdida
from_state,,,,,,,,,,,,,
Solicitação de material,0.0,0.588,0.002,0.000,0.000,0.000,0.00,0.0,0.000,0.000,0.000,0.000,0.410
Material entregue,0.0,0.000,0.908,0.002,0.000,0.000,0.00,0.0,0.000,0.000,0.000,0.000,0.090
Validação pedagógica,0.0,0.000,0.000,0.470,0.000,0.000,0.00,0.0,0.000,0.000,0.000,0.000,0.530
Proposta tabela,0.0,0.000,0.004,0.000,0.704,0.000,0.00,0.0,0.000,0.000,0.000,0.000,0.292
Proposta personalizada,0.0,0.000,0.000,0.000,0.000,0.491,0.00,0.0,0.000,0.000,0.000,0.000,0.509
Negociação e ajuste financeiro,0.0,0.000,0.000,0.000,0.000,0.000,0.75,0.0,0.000,0.000,0.000,0.000,0.250
Documentação enviada,0.0,0.000,0.000,0.000,0.000,0.000,0.00,1.0,0.000,0.000,0.000,0.000,0.000
Análise de operações comerciais,0.0,0.000,0.000,0.000,0.000,0.000,0.00,0.0,0.000,0.905,0.000,0.000,0.095
Análise do Financeiro,0.0,0.000,0.000,0.000,0.000,0.000,0.00,0.0,0.000,0.000,0.983,0.000,0.017


Somas das linhas:


,row_sum
from_state,
Solicitação de material,1.0
Material entregue,1.0
Validação pedagógica,1.0
Proposta tabela,1.0
Proposta personalizada,1.0
Negociação e ajuste financeiro,1.0
Documentação enviada,1.0
Análise de operações comerciais,1.0
Análise do Financeiro,1.0


,Contrato assinado,Oportunidade perdida
Solicitação de material,0.0546,0.9454
Material entregue,0.0924,0.9076
Validação pedagógica,0.1014,0.8986
Proposta tabela,0.2158,0.7842
Proposta personalizada,0.3060,0.6940
Negociação e ajuste financeiro,0.6230,0.3770
Documentação enviada,0.8306,0.1694
Análise de operações comerciais,0.8306,0.1694
Análise do Financeiro,0.9333,0.0667
Análise do Jurídico,0.9180,0.0820


,expected_transitions
Solicitação de material,2.95
Material entregue,3.31
Validação pedagógica,2.54
Proposta tabela,3.27
Proposta personalizada,3.21
Negociação e ajuste financeiro,4.50
Documentação enviada,4.67
Análise de operações comerciais,3.67
Análise do Financeiro,1.98
Análise do Jurídico,2.95


In [6]:
# Exemplo de saída da camada Markov para oportunidades abertas.
if not absorption.empty:
    markov_open = open_[['opportunity_id','account_id','brand','current_stage','potential_revenue']].copy()
    markov_open['markov_prob_won'] = markov_open['current_stage'].map(absorption[ABSORBING_WON])
    markov_open['markov_prob_lost'] = markov_open['current_stage'].map(absorption[ABSORBING_LOST])
    markov_open = markov_open.sort_values('markov_prob_won', ascending=False, na_position='last')
    print('Exemplo: dez oportunidades abertas ordenadas pelo baseline Markov')
    display(markov_open.head(10))
    display(markov_open['markov_prob_won'].describe())
else:
    markov_open = pd.DataFrame()
    print('Sem previsão Markov disponível.')

Exemplo: dez oportunidades abertas ordenadas pelo baseline Markov


,opportunity_id,account_id,brand,current_stage,potential_revenue,markov_prob_won,markov_prob_lost
93,006N400000WvgjFIAR,001N400000c3vAUIAY,Poliedro,Enviado para a assinatura,75783.10,0.949153,0.050847
1113,006N400000dKoOqIAK,001N400000NlF3XIAV,Poliedro,Enviado para a assinatura,419817.18,0.949153,0.050847
1196,006N400000eT0lcIAC,001N400000qWAxeIAG,Poliedro,Enviado para a assinatura,307629.44,0.949153,0.050847
1452,006N400000jcG3SIAU,001N400000vY2t0IAC,Polígono,Enviado para a assinatura,121096.02,0.949153,0.050847
152,006N400000YyjGQIAZ,001N4000009EeICIA0,Conviver,Enviado para a assinatura,152925.00,0.949153,0.050847
1461,006N400000juwicIAA,001N400000heTzCIAU,Poliedro,Enviado para a assinatura,166714.27,0.949153,0.050847
1542,006N400000mm9DRIAY,0014v00002ZitaVAAR,Poliedro,Enviado para a assinatura,268017.98,0.949153,0.050847
763,006N400000aGtrsIAC,001N400000jgLWAIA2,Polígono,Enviado para a assinatura,53448.79,0.949153,0.050847
11,006N400000SqJPpIAN,0014v00002qjwSUAAY,Polígono,Enviado para a assinatura,1184003.90,0.949153,0.050847
1197,006N400000eTtWjIAK,001N400000qXP07IAG,Poliedro,Enviado para a assinatura,715546.15,0.949153,0.050847


count    656.000000
mean       0.191193
std        0.199014
min        0.054556
25%        0.054556
50%        0.101385
75%        0.215758
max        0.949153
Name: markov_prob_won, dtype: float64

## 4. Enriquecimento da oportunidade para modelagem individual

As tabelas auxiliares são juntadas por chaves existentes no arquivo. A junção da base de escolas e das campanhas usa `account_id`; a base de especialistas usa `opportunity_id`. A base de perdas também possui `account_id`, portanto é mantida como informação auxiliar e não é usada automaticamente como rótulo da oportunidade. O desfecho principal é derivado do último estado do histórico do funil.

Essa escolha evita afirmar que um registro de perda em nível de conta seja necessariamente a perda de uma oportunidade específica.

In [7]:
# Agregação da escola: a auditoria deve confirmar se há uma linha única por account_id.
print('Contas duplicadas na base de escolas:', int(escolas['account_id'].duplicated().sum()))
escolas_one = escolas.drop_duplicates('account_id', keep='first').copy()

model_df = journeys.merge(escolas_one, on='account_id', how='left', suffixes=('','_school'), validate='many_to_one')
model_df = model_df.merge(esp_summary, on='opportunity_id', how='left', validate='one_to_one')
model_df = model_df.merge(camp_summary, on='account_id', how='left', validate='many_to_one')
model_df = model_df.merge(loss_summary, on='account_id', how='left', validate='many_to_one')

model_df['target_won'] = model_df['outcome'].map({'won':1, 'lost':0})
model_df['age_days_at_last_event'] = (model_df['last_event_date'] - model_df['first_event_date']).dt.days
model_df['stage_days_proxy'] = model_df['duration_days']
model_df['n_advances_proxy'] = model_df.apply(lambda r: sum((seq_map and False) for _ in []), axis=1) if False else np.nan

print('Modelo enriquecido:', model_df.shape)
coverage = pd.DataFrame({
    'campo': ['school_profile','brand','consultant_id','state','cluster','potential_revenue','specialist_records','campaign_count'],
    'cobertura': [model_df[c].notna().mean() if c in model_df else np.nan for c in ['school_profile','brand','consultant_id','state','cluster','potential_revenue','specialist_records','campaign_count']]
})
display(coverage)
display(model_df.head(3))

Contas duplicadas na base de escolas: 1102
Modelo enriquecido: (1610, 46)


,campo,cobertura
0,school_profile,0.709317
1,brand,0.994410
2,consultant_id,0.709317
3,state,0.709317
4,cluster,0.709317
5,potential_revenue,1.000000
6,specialist_records,0.998758
7,campaign_count,0.313043


,opportunity_id,account_id,brand,potential_revenue,created_at,first_event_date,last_event_date,stage_sequence,status_sequence,event_count,current_stage,unique_stage_count,transition_count,duration_days,outcome,has_missing_dates,school_profile,school_size,cluster,score,priority,students_efaf,students_efai,students_ei,students_em,city,state,lead_source,religion,network,consultant_id,average_tuition,contract_expiry,current_system,completed_visits,specialist_records,specialist_actions,campaign_count,campaign_parents,loss_records,loss_date_aux,loss_reason_aux,target_won,age_days_at_last_event,stage_days_proxy,n_advances_proxy
0,006N400000AZoSRIA1,001N400000C8XpZIAV,Polígono,161441.00,2026-03-11 02:09:23,2026-03-11,2026-04-27,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",3,Validação pedagógica,3,2,47,open,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1 - Sim,NaN,NaN,1.0,2026-04-27,,NaN,47,47,NaN
1,006N400000C3EFeIAN,001N400000DxyByIAJ,Poliedro,949049.12,2026-03-17 18:32:40,2026-03-17,2026-06-15,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",4,Oportunidade perdida,4,3,90,lost,False,Prospecção,750-1.000,C - Estruturação,575.0,Alta,277.0,309.0,62.0,127.0,São Paulo,SP,Censo Escolar 2021,Católica,NaN,0056g0000034OkkAAE,R$ 2.000 a R$ 2.500,2027,FTD,3.0,1.0,3 - Não,NaN,NaN,1.0,2026-06-15,Satisfação com serviço do atual parceiro,0.0,90,90,NaN
2,006N400000KkLAnIAN,0014v00002uO7OMAA0,Poliedro,595118.69,2026-03-04 15:31:14,2026-03-04,2026-03-18,"[Solicitação de material, Material entregue, V...","[Avançou para a Próxima Etapa, Avançou para a ...",12,Contrato assinado,12,11,14,won,False,Prospecção,Acima de 1.000,A - Referência,825.0,Alta,562.0,737.0,112.0,288.0,Campinas,SP,Censo Escolar 2021,Evangélica,NaN,0054v00000EkXMDAA3,R$ 500 a R$ 1.000,2026,Positivo,19.0,1.0,1 - Sim,NaN,NaN,1.0,2026-03-18,,1.0,14,14,NaN


## 5. Camada 3 — Propensão à assinatura

O exemplo usa uma regressão logística regularizada, com codificação one-hot para variáveis categóricas e imputação para valores ausentes. A divisão é temporal: oportunidades mais antigas treinam o modelo e oportunidades posteriores testam o modelo.

O conjunto de preditores é deliberadamente conservador. Campos que representam desfechos posteriores, como `loss_reason_aux` e `loss_date_aux`, são excluídos. A coluna `current_stage` é usada apenas no modelo operacional; o modelo estrutural usa características do perfil e do contexto.

Se o número de contratos ganhos for insuficiente para uma divisão temporal com duas classes, o notebook não força o ajuste e registra a limitação.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def temporal_split(df, date_col='first_event_date', test_fraction=0.25):
    d = df.dropna(subset=[date_col]).sort_values(date_col).copy()
    if len(d) < 4:
        return d.iloc[0:0], d.iloc[0:0], None
    cut_idx = max(1, min(len(d)-1, int(len(d)*(1-test_fraction))))
    cut_date = d.iloc[cut_idx][date_col]
    return d[d[date_col] < cut_date].copy(), d[d[date_col] >= cut_date].copy(), cut_date

structural = ['school_profile','school_size','cluster','score','priority','students_efaf','students_efai','students_ei','students_em','state','lead_source','religion','network','average_tuition','contract_expiry','current_system','brand','potential_revenue']
operational = structural + ['current_stage','duration_days','event_count','unique_stage_count','transition_count','completed_visits','specialist_records','campaign_count']
# Somente colunas existentes entram; nada é criado para preencher ausência estrutural.

def fit_propensity(df, features, label):
    use = [c for c in features if c in df.columns]
    d = df.dropna(subset=['target_won','first_event_date']).copy()
    train, test, cut = temporal_split(d)
    result = {'label': label, 'features': use, 'cut_date': cut, 'model': None, 'predictions': pd.DataFrame()}
    if train.empty or test.empty or train['target_won'].nunique()<2 or test['target_won'].nunique()<2:
        result['reason'] = 'Não há duas classes em treino e teste após o corte temporal.'
        return result
    X_train, y_train = train[use], train['target_won'].astype(int)
    X_test, y_test = test[use], test['target_won'].astype(int)
    num = X_train.select_dtypes(include=np.number).columns.tolist()
    cat = [c for c in use if c not in num]
    prep = ColumnTransformer([
        ('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num),
        ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat)
    ])
    pipe = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:,1]
    result.update(model=pipe, predictions=pd.DataFrame({'opportunity_id':test['opportunity_id'].values,'observed':y_test.values,'predicted_prob':p}),
                  auc=roc_auc_score(y_test,p), average_precision=average_precision_score(y_test,p), brier=brier_score_loss(y_test,p), train_n=len(train), test_n=len(test))
    return result

propensity_results = {}
for label, features in [('structural', structural), ('operational', operational)]:
    propensity_results[label] = fit_propensity(model_df, features, label)
    r = propensity_results[label]
    print('Modelo:', label, '| variáveis:', len(r['features']), '| corte:', r.get('cut_date'))
    if r.get('model') is None:
        print('Limitação:', r.get('reason'))
    else:
        print('treino:', r['train_n'], '| teste:', r['test_n'], '| AUC:', round(r['auc'],4), '| AP:', round(r['average_precision'],4), '| Brier:', round(r['brier'],4))
        display(r['predictions'].sort_values('predicted_prob', ascending=False).head(10))

Modelo: structural | variáveis: 18 | corte: 2026-04-17 00:00:00
treino: 708 | teste: 245 | AUC: 0.838 | AP: 0.3011 | Brier: 0.1277


,opportunity_id,observed,predicted_prob
75,006N400000eD37WIAS,0,0.999713
100,006N400000el7obIAA,0,0.998998
230,006N400000kfIcqIAE,1,0.995242
85,006N400000eMkV8IAK,0,0.995210
221,006N400000jOSuAIAW,1,0.993273
130,006N400000f9s3CIAQ,1,0.991768
58,006N400000dhEUHIA2,0,0.989147
243,006N400000miQpZIAU,1,0.985600
3,006N400000cuzpLIAQ,0,0.984765
229,006N400000ke4GwIAI,1,0.983318


Modelo: operational | variáveis: 26 | corte: 2026-04-17 00:00:00
treino: 708 | teste: 245 | AUC: 1.0 | AP: 1.0 | Brier: 0.0027


,opportunity_id,observed,predicted_prob
194,006N400000hyoZ5IAI,1,0.999824
230,006N400000kfIcqIAE,1,0.999690
228,006N400000kfbE3IAI,1,0.999653
238,006N400000lpK4uIAE,1,0.999585
130,006N400000f9s3CIAQ,1,0.999440
221,006N400000jOSuAIAW,1,0.999368
229,006N400000ke4GwIAI,1,0.999283
155,006N400000ftdKYIAY,1,0.998194
243,006N400000miQpZIAU,1,0.997112
138,006N400000fWjX3IAK,1,0.996111


### Interpretação da comparação estrutural versus operacional

A comparação só deve ser feita depois de verificar o corte temporal, a composição das classes e a cobertura das variáveis. Se o modelo operacional superar o estrutural, isso sugere que a trajetória recente acrescenta informação preditiva além do fit estrutural. Isso é uma associação preditiva, não uma prova de que visitas ou ações comerciais causam conversão.

In [9]:
# Aplicação do melhor modelo disponível às oportunidades abertas, sem usar desfechos posteriores.
open_predictions = open_[['opportunity_id']].merge(model_df, on='opportunity_id', how='left', suffixes=('','_model'))
for label, result in propensity_results.items():
    if result.get('model') is None:
        continue
    features = result['features']
    open_predictions[f'prob_won_{label}'] = result['model'].predict_proba(open_predictions[features])[:,1]

prob_cols = [c for c in open_predictions if c.startswith('prob_won_')]
if prob_cols:
    display(open_predictions[['opportunity_id','current_stage','brand'] + prob_cols].sort_values(prob_cols[-1], ascending=False).head(10))
else:
    print('Não foi possível gerar previsão individual para oportunidades abertas com o corte atual.')

,opportunity_id,current_stage,brand,prob_won_structural,prob_won_operational
250,006N400000ah51LIAQ,Contrato Ativo,Polígono,0.313368,0.994376
373,006N400000eT0lcIAC,Enviado para a assinatura,Poliedro,0.988044,0.987069
47,006N400000YyjGQIAZ,Enviado para a assinatura,Conviver,0.942937,0.984376
590,006N400000mm9DRIAY,Enviado para a assinatura,Poliedro,0.952070,0.979596
374,006N400000eTtWjIAK,Enviado para a assinatura,Poliedro,0.975279,0.979212
222,006N400000aGtrsIAC,Enviado para a assinatura,Polígono,0.872525,0.960772
340,006N400000dKoOqIAK,Enviado para a assinatura,Poliedro,0.148303,0.957682
2,006N400000SqJPpIAN,Enviado para a assinatura,Polígono,0.292254,0.945952
330,006N400000cqFmjIAE,Enviado para a assinatura,Polígono,0.061039,0.943355
525,006N400000juwicIAA,Enviado para a assinatura,Poliedro,0.023559,0.934925


## 6. Camada temporal exploratória

O dataset contém datas de eventos e permite calcular a duração observada entre o primeiro e o último registro da oportunidade. Nesta etapa, o notebook demonstra a estrutura necessária para tempo até o evento e apresenta estatísticas por desfecho.

A duração observada não deve ser interpretada automaticamente como tempo até assinatura: para oportunidades abertas, ela é censurada; para oportunidades perdidas, o evento é perda. A adoção de um modelo de riscos competitivos depende de confirmar que as datas representam adequadamente entrada, movimentação e encerramento.

In [10]:
temporal_summary = (model_df.groupby('outcome', dropna=False)
    .agg(n=('opportunity_id','size'), median_duration_days=('duration_days','median'), mean_duration_days=('duration_days','mean'),
         min_duration_days=('duration_days','min'), max_duration_days=('duration_days','max'))
    .reset_index())
display(temporal_summary)

# Probabilidade empírica de assinatura por etapa atual entre oportunidades finalizadas.
stage_outcome = (finished.assign(current_stage=finished['current_stage'])
    .groupby('current_stage')['outcome'].value_counts(normalize=True).unstack(fill_value=0))
for col in ['won','lost']:
    if col not in stage_outcome: stage_outcome[col] = 0.0
stage_outcome = stage_outcome.rename(columns={'won':'empirical_p_won','lost':'empirical_p_lost'})
display(stage_outcome.sort_values('empirical_p_won', ascending=False))

,outcome,n,median_duration_days,mean_duration_days,min_duration_days,max_duration_days
0,lost,897,77.0,77.810479,0,241
1,open,657,32.0,39.557078,0,163
2,won,56,39.5,64.839286,0,159


outcome,empirical_p_lost,empirical_p_won
current_stage,,
Contrato assinado,0.0,1.0
Oportunidade perdida,1.0,0.0


### Opção de tempo discreto

Para uma implementação final, cada oportunidade deve ser expandida em intervalos semanais ou mensais até assinatura, perda ou censura. Em cada intervalo, o alvo pode distinguir: assinatura, perda ou permanência. Os preditores precisam ser recalculados no início de cada intervalo. Este notebook não executa essa expansão automaticamente porque a granularidade temporal e o significado de cada evento precisam ser confirmados no projeto.

In [11]:
# Esqueleto verificável para a futura expansão de intervalos; não cria dados nem ajusta modelo.
def temporal_model_requirements(df):
    required = ['opportunity_id','first_event_date','last_event_date','outcome']
    return {
        'required_columns_present': {c: c in df.columns for c in required},
        'outcomes_present': sorted(df['outcome'].dropna().unique().tolist()),
        'date_coverage': df[['first_event_date','last_event_date']].notna().mean().to_dict(),
        'can_define_censoring': set(df['outcome'].dropna().unique()).issuperset({'open'}),
        'warning': 'Confirmar se last_event_date é encerramento para ganhas/perdidas e data de corte para abertas.'
    }
requirements = temporal_model_requirements(model_df)
display(pd.Series(requirements, dtype='object').to_frame('status'))

,status
required_columns_present,"{'opportunity_id': True, 'first_event_date': T..."
outcomes_present,"[lost, open, won]"
date_coverage,"{'first_event_date': 1.0, 'last_event_date': 1.0}"
can_define_censoring,True
warning,Confirmar se last_event_date é encerramento pa...


## 7. Camada de prioridade comercial

A priorização abaixo é uma demonstração. Ela não fixa pontos de corte nem cria categorias de negócio sem autorização. Quando existir uma probabilidade individual válida e receita potencial disponível, calcula-se o valor esperado simples. O uso de tempo e confiança deve ser definido após a camada temporal e a validação.

In [12]:
# Usa a probabilidade operacional quando disponível; caso contrário, Markov como referência, sem misturar silenciosamente os modelos.
priority_demo = model_df[['opportunity_id','brand','current_stage','potential_revenue','outcome']].copy()
if 'prob_won_operational' in open_predictions.columns:
    priority_demo = priority_demo.merge(open_predictions[['opportunity_id','prob_won_operational']], on='opportunity_id', how='left')
else:
    priority_demo['prob_won_operational'] = np.nan
if not markov_open.empty:
    priority_demo = priority_demo.merge(markov_open[['opportunity_id','markov_prob_won']], on='opportunity_id', how='left')
else:
    priority_demo['markov_prob_won'] = np.nan
priority_demo['expected_revenue_operational'] = priority_demo['potential_revenue'] * priority_demo['prob_won_operational']
priority_demo['expected_revenue_markov'] = priority_demo['potential_revenue'] * priority_demo['markov_prob_won']
print('O valor esperado é demonstrativo e não substitui um índice final com prazo e confiança.')
display(priority_demo[priority_demo['outcome'].eq('open')].sort_values('expected_revenue_operational', ascending=False, na_position='last').head(10))

O valor esperado é demonstrativo e não substitui um índice final com prazo e confiança.


,opportunity_id,brand,current_stage,potential_revenue,outcome,prob_won_operational,markov_prob_won,expected_revenue_operational,expected_revenue_markov
11,006N400000SqJPpIAN,Polígono,Enviado para a assinatura,1184003.90,open,0.945952,0.949153,1.120011e+06,1.123800e+06
1197,006N400000eTtWjIAK,Poliedro,Enviado para a assinatura,715546.15,open,0.979212,0.949153,7.006717e+05,6.791624e+05
1113,006N400000dKoOqIAK,Poliedro,Enviado para a assinatura,419817.18,open,0.957682,0.949153,4.020515e+05,3.984705e+05
1196,006N400000eT0lcIAC,Poliedro,Enviado para a assinatura,307629.44,open,0.987069,0.949153,3.036515e+05,2.919873e+05
1593,006N400000o725EIAQ,Polígono,Análise do Jurídico,371415.58,open,0.710806,0.918033,2.640046e+05,3.409717e+05
1542,006N400000mm9DRIAY,Poliedro,Enviado para a assinatura,268017.98,open,0.979596,0.949153,2.625494e+05,2.543899e+05
835,006N400000ah51LIAQ,Polígono,Contrato Ativo,225102.91,open,0.994376,NaN,2.238370e+05,NaN
1461,006N400000juwicIAA,Poliedro,Enviado para a assinatura,166714.27,open,0.934925,0.949153,1.558654e+05,1.582373e+05
152,006N400000YyjGQIAZ,Conviver,Enviado para a assinatura,152925.00,open,0.984376,0.949153,1.505358e+05,1.451492e+05
949,006N400000bWx7iIAC,Poliedro,Documentação enviada,1950612.43,open,0.072845,0.830601,1.420924e+05,1.620181e+06


## 8. Testes de exemplo e saídas

Os exemplos abaixo não selecionam uma oportunidade por conveniência analítica. Eles mostram: (a) algumas oportunidades finalizadas para inspeção da jornada; (b) algumas oportunidades abertas para testar a aplicação das previsões; e (c) um caso por marca quando houver observações disponíveis. Os identificadores são mantidos apenas para rastreabilidade interna.

In [13]:
# Exemplo 1: jornadas finalizadas selecionadas por ordem temporal, sem amostragem aleatória.
finished_examples = finished.sort_values('first_event_date').head(5)[['opportunity_id','brand','current_stage','outcome','event_count','duration_days','stage_sequence']]
display(finished_examples)

# Exemplo 2: oportunidades abertas mais recentes, se existirem.
open_examples = open_.sort_values('last_event_date', ascending=False).head(5)[['opportunity_id','brand','current_stage','event_count','duration_days','stage_sequence']]
display(open_examples)

# Exemplo 3: uma oportunidade finalizada por marca, somente onde houver dados.
by_brand_examples = (finished.sort_values('first_event_date')
    .groupby('brand', dropna=False, as_index=False).head(1)
    [['opportunity_id','brand','outcome','current_stage','duration_days']])
display(by_brand_examples)

,opportunity_id,brand,current_stage,outcome,event_count,duration_days,stage_sequence
18,006N400000TXKHAIA5,Poliedro,Oportunidade perdida,lost,2,241,"[Solicitação de material, Oportunidade perdida]"
30,006N400000UsiPYIAZ,Poliedro,Oportunidade perdida,lost,3,187,"[Solicitação de material, Validação pedagógica..."
79,006N400000Wtf07IAB,Poliedro,Oportunidade perdida,lost,2,201,"[Solicitação de material, Oportunidade perdida]"
80,006N400000Wtj5KIAR,NaN,Oportunidade perdida,lost,2,169,"[Solicitação de material, Oportunidade perdida]"
118,006N400000XAI0fIAH,NaN,Oportunidade perdida,lost,2,188,"[Solicitação de material, Oportunidade perdida]"


,opportunity_id,brand,current_stage,event_count,duration_days,stage_sequence
1609,006N400000oUy5BIAS,Poliedro,Solicitação de material,1,0,[Solicitação de material]
3,006N400000Lv4yrIAB,Poliedro,Solicitação de material,1,0,[Solicitação de material]
1460,006N400000jup2mIAA,Poliedro,Negociação e ajuste financeiro,6,44,"[Solicitação de material, Material entregue, V..."
1418,006N400000ieIZgIAM,Poliedro,Proposta tabela,4,57,"[Solicitação de material, Material entregue, V..."
1373,006N400000hjM24IAE,Polígono,Negociação e ajuste financeiro,6,65,"[Solicitação de material, Material entregue, V..."


,opportunity_id,brand,outcome,current_stage,duration_days
18,006N400000TXKHAIA5,Poliedro,lost,Oportunidade perdida,241
80,006N400000Wtj5KIAR,NaN,lost,Oportunidade perdida,169
476,006N400000ZybqOIAR,Conviver,lost,Oportunidade perdida,92
324,006N400000ZrNQaIAN,Polígono,lost,Oportunidade perdida,103


## 9. Limitações registradas e próximos passos

1. A cadeia de Markov atual é homogênea e não incorpora perfil individual.
2. A junção de perdas por conta é auxiliar e não é tratada como rótulo de oportunidade.
3. A definição de `current_stage` depende do último evento registrado, que precisa ser validado contra a semântica do CRM.
4. A duração entre primeiro e último evento não é automaticamente o tempo real até assinatura ou perda.
5. A base contém desfechos muito desbalanceados; modelos individuais precisam de avaliação de estabilidade e incerteza.
6. O notebook não interpreta associações como efeitos causais.
7. A modelagem final deve comparar o baseline Markov, a propensão estrutural, a propensão operacional e o componente temporal em cortes futuros.
8. Somente após a validação devem ser definidos horizontes, faixas de prioridade, rotina de atualização e regras de ação comercial.

In [16]:
# Exportação dos principais artefatos exploratórios.
transition.to_csv(OUTPUT_DIR/'baseline_matriz_transicao.csv')
if not absorption.empty:
    absorption.to_csv(OUTPUT_DIR/'baseline_probabilidade_absorcao.csv')
expected_steps.to_csv(OUTPUT_DIR/'baseline_transicoes_esperadas.csv')
model_df.to_csv(OUTPUT_DIR/'oportunidades_enriquecidas_exploratorio.csv', index=False)
if not markov_open.empty:
    markov_open.to_csv(OUTPUT_DIR/'previsoes_abertas_markov.csv', index=False)
for label, result in propensity_results.items():
    if not result.get('predictions', pd.DataFrame()).empty:
        result['predictions'].to_csv(OUTPUT_DIR/f'validacao_propensao_{label}.csv', index=False)
print('Artefatos exportados em:', OUTPUT_DIR)

Artefatos exportados em: resultados-v1
